In [41]:
import pandas as pd
import numpy as np

# Introdução ao groupby e agregações no Pandas
O groupby é um dos recursos mais poderosos do Pandas.
Ele permite agrupar dados por categorias e aplicar operações de agregação,
transformação e análise em cada grupo.
Neste notebook vamos explorar:
- groupby simples
- groupby com múltiplas colunas
- groupby com `apply()`
- Diferença entre `agg()` e `transform()`
- Uso de dropna=False
- pivot_table
- crosstab

In [42]:
dados = pd.DataFrame({
    "departamento": ["TI", "TI", "RH", "RH", "Financeiro", "Financeiro", np.nan],
    "cidade": ["Lisboa", "Porto", "Lisboa", "Porto", "Lisboa", "Porto", "Lisboa"],
    "funcionario": ["Ana", "Bruno", "Carlos", "Daniela", "Eduardo", "Fernanda", "Gabriel"],
    "salario": [4000, 4500, 3500, 3600, 5000, 5200, 3000],
    "idade": [28, 32, 41, 38, 45, 29, 31]
})
dados

,departamento,cidade,funcionario,salario,idade
0,TI,Lisboa,Ana,4000,28
1,TI,Porto,Bruno,4500,32
2,RH,Lisboa,Carlos,3500,41
3,RH,Porto,Daniela,3600,38
4,Financeiro,Lisboa,Eduardo,5000,45
5,Financeiro,Porto,Fernanda,5200,29
6,NaN,Lisboa,Gabriel,3000,31


# 1. GroupBy simples
Aqui agrupamos os dados por departamento
e calculamos a média salarial de cada grupo.

In [43]:
media_salario = dados.groupby("departamento")["salario"].mean()
media_salario

departamento
Financeiro    5100.0
RH            3550.0
TI            4250.0
Name: salario, dtype: float64

Também podemos aplicar múltiplas agregações ao mesmo tempo.

In [44]:
resumo_departamento = (
    dados.groupby("departamento")
    .agg({
        "salario": ["mean", "max", "min"],
        "idade": "mean"
    })
)
resumo_departamento

salario             idade
                mean   max   min  mean
departamento                          
Financeiro    5100.0  5200  5000  37.0
RH            3550.0  3600  3500  39.5
TI            4250.0  4500  4000  30.0

# 2. GroupBy em mais de uma coluna
Podemos agrupar utilizando múltiplas dimensões.
Neste exemplo vamos agrupar por:
- departamento
- cidade

In [45]:
grupo_duplo = (
    dados.groupby(["departamento", "cidade"])["salario"].mean()
)
grupo_duplo

departamento  cidade
Financeiro    Lisboa    5000.0
              Porto     5200.0
RH            Lisboa    3500.0
              Porto     3600.0
TI            Lisboa    4000.0
              Porto     4500.0
Name: salario, dtype: float64

O resultado possui um índice hierárquico (MultiIndex).

In [46]:
grupo_duplo.reset_index()

,departamento,cidade,salario
0,Financeiro,Lisboa,5000.0
1,Financeiro,Porto,5200.0
2,RH,Lisboa,3500.0
3,RH,Porto,3600.0
4,TI,Lisboa,4000.0
5,TI,Porto,4500.0


# 3. GroupBy utilizando `apply()` com função externa
O `apply()` permite executar funções customizadas
em cada grupo.

In [47]:
SALARIO_MINIMO = 500

def classificar_salario(grupo):

    grupo = grupo.copy()

    # cálculo da razão salarial
    grupo["ratio_salario_minimo"] = (
        grupo["salario"] / SALARIO_MINIMO
    )

    # classificação categórica
    grupo["faixa_salarial"] = np.where(
        grupo["ratio_salario_minimo"] < 8,
        "baixo",
        np.where(
            grupo["ratio_salario_minimo"] < 10,
            "medio",
            "alto"
        )
    )

    return grupo

# %%
resultado_apply = (
    dados.groupby("departamento")
    .apply(classificar_salario)
)

resultado_apply

cidade funcionario  salario  idade  ratio_salario_minimo  \
departamento                                                               
Financeiro   4  Lisboa     Eduardo     5000     45                  10.0   
             5   Porto    Fernanda     5200     29                  10.4   
RH           2  Lisboa      Carlos     3500     41                   7.0   
             3   Porto     Daniela     3600     38                   7.2   
TI           0  Lisboa         Ana     4000     28                   8.0   
             1   Porto       Bruno     4500     32                   9.0   

               faixa_salarial  
departamento                   
Financeiro   4           alto  
             5           alto  
RH           2          baixo  
             3          baixo  
TI           0          medio  
             1          medio

# 4. Diferença entre agg() e transform()
Embora os dois sejam usados após um groupby,
eles possuem objetivos diferentes.
## agg()
Retorna uma versão resumida dos grupos.
## transform()
Retorna um resultado com o mesmo tamanho do DataFrame original.
%% [markdown]
## Exemplo com agg()

In [48]:
resultado_agg = (
    dados.groupby("departamento")["salario"]
    .agg("mean")
)
resultado_agg

departamento
Financeiro    5100.0
RH            3550.0
TI            4250.0
Name: salario, dtype: float64

O resultado possui uma linha por grupo.

In [49]:
dados["media_departamento"] = (
    dados.groupby("departamento")["salario"]
    .transform("mean")
)
dados

,departamento,cidade,funcionario,salario,idade,media_departamento
0,TI,Lisboa,Ana,4000,28,4250.0
1,TI,Porto,Bruno,4500,32,4250.0
2,RH,Lisboa,Carlos,3500,41,3550.0
3,RH,Porto,Daniela,3600,38,3550.0
4,Financeiro,Lisboa,Eduardo,5000,45,5100.0
5,Financeiro,Porto,Fernanda,5200,29,5100.0
6,NaN,Lisboa,Gabriel,3000,31,NaN


Agora temos:
- Uma média calculada por departamento
- Repetida para cada linha correspondente
- Mantendo o mesmo número de linhas do DataFrame original

Podemos usar isso para criar métricas linha a linha.

In [50]:
dados["salario_vs_media"] = (
    dados["salario"] - dados["media_departamento"]
)
dados

,departamento,cidade,funcionario,salario,idade,media_departamento,salario_vs_media
0,TI,Lisboa,Ana,4000,28,4250.0,-250.0
1,TI,Porto,Bruno,4500,32,4250.0,250.0
2,RH,Lisboa,Carlos,3500,41,3550.0,-50.0
3,RH,Porto,Daniela,3600,38,3550.0,50.0
4,Financeiro,Lisboa,Eduardo,5000,45,5100.0,-100.0
5,Financeiro,Porto,Fernanda,5200,29,5100.0,100.0
6,NaN,Lisboa,Gabriel,3000,31,NaN,NaN


# 5. groupby() com dropna=False
Por padrão, grupos com valores NaN
são ignorados.
Utilizando dropna=False,
conseguimos incluir esses registros no agrupamento.

In [51]:
dados.groupby("departamento").size()

departamento
Financeiro    2
RH            2
TI            2
dtype: int64

Observe que o grupo NaN não aparece.

In [52]:
dados.groupby("departamento", dropna=False).size()

departamento
Financeiro    2
RH            2
TI            2
NaN           1
dtype: int64

Agora os valores ausentes foram considerados no agrupamento.

# 6. Pivot Tables
pivot_table é muito útil para análises multidimensionais,
semelhantes a tabelas dinâmicas do Excel.

In [53]:
dados

,departamento,cidade,funcionario,salario,idade,media_departamento,salario_vs_media
0,TI,Lisboa,Ana,4000,28,4250.0,-250.0
1,TI,Porto,Bruno,4500,32,4250.0,250.0
2,RH,Lisboa,Carlos,3500,41,3550.0,-50.0
3,RH,Porto,Daniela,3600,38,3550.0,50.0
4,Financeiro,Lisboa,Eduardo,5000,45,5100.0,-100.0
5,Financeiro,Porto,Fernanda,5200,29,5100.0,100.0
6,NaN,Lisboa,Gabriel,3000,31,NaN,NaN


In [54]:
pivot_salarios = pd.pivot_table(
    dados,
    values="salario",
    index="departamento",
    columns="cidade",
    aggfunc="mean"
)
pivot_salarios

cidade,Lisboa,Porto
departamento,,
Financeiro,5000.0,5200.0
RH,3500.0,3600.0
TI,4000.0,4500.0


Também podemos usar múltiplas agregações.

In [55]:
pivot_multi = pd.pivot_table(
    dados,
    values="salario",
    index="departamento",
    columns="cidade",
    aggfunc=["mean", "max", "min"]
)
pivot_multi

mean            max          min      
cidade        Lisboa   Porto Lisboa Porto Lisboa Porto
departamento                                          
Financeiro    5000.0  5200.0   5000  5200   5000  5200
RH            3500.0  3600.0   3500  3600   3500  3600
TI            4000.0  4500.0   4000  4500   4000  4500

# 7. Crosstab
crosstab é utilizado para gerar tabelas de frequência
entre variáveis categóricas.

In [56]:
crosstab_departamento = pd.crosstab(
    dados["departamento"],
    dados["cidade"]
)
crosstab_departamento

cidade,Lisboa,Porto
departamento,,
Financeiro,1,1
RH,1,1
TI,1,1


Também podemos normalizar os valores
para obter proporções.

In [57]:
crosstab_normalizado = pd.crosstab(
    dados["departamento"],
    dados["cidade"],
    normalize="index" #aqui
)
crosstab_normalizado

cidade,Lisboa,Porto
departamento,,
Financeiro,0.5,0.5
RH,0.5,0.5
TI,0.5,0.5


# Conclusão
Neste notebook exploramos:
- Agrupamentos simples e múltiplos
- Uso de apply()
- Diferenças entre agg() e transform()
- Inclusão de valores ausentes com dropna=False
- pivot_table
- crosstab
Esses recursos são fundamentais para análise de dados com Pandas.

In [73]:
#read txt from ./data folder
tips = pd.read_csv("./data/tip.csv")
tips

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


### Desafio
Considerando as tabelas ecomerce e clientes responda:
1. Em que horário refeição se paga melhor as tips
2. Que genero paga a maior taxa `(tip/total_bill)`
3. Quem fuma mais? Homens ou mulheres?
4. Qual é o dia e horário que se paga a maior taxa?
5. Considerando que as mesas do restaurante tem 4 lugares. Qual genero ocupou mais de 2 no mesmo tempo?
6. Qual é o valor médio da gorjeta (tip) por dia da semana e por período (Lunch ou Dinner)?(use pivot table)
